# Анализ результатов CNN vs ViT

Notebook соответствует структуре экспериментальной главы ВКР: сначала анализируются доменные протоколы Office-Home (`in_domain`, `cross_domain`, `lodo`, `semantic_ood`), затем устойчивость к common corruptions, калибровка, OOD-поведение и вычислительная эффективность.

Входные файлы:

- `outputs/tables/summary.csv`
- `outputs/tables/metrics_long.csv`
- `outputs/tables/lodo_summary.csv`
- `outputs/tables/acs_full.csv`

Фигуры сохраняются в `outputs/figures/`.

In [ ]:
from __future__ import annotations

import csv
import json
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
OUT = ROOT / 'outputs'
TABLES = OUT / 'tables'
METRICS = OUT / 'metrics'
FIGURES = OUT / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 220,
    'font.size': 10,
    'axes.titlesize': 12,
    'axes.labelsize': 10,
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linewidth': 0.8,
})

PROTOCOLS = ['in_domain', 'cross_domain', 'lodo', 'semantic_ood']
PROTOCOL_LABEL = {
    'in_domain': 'In-domain',
    'cross_domain': 'Cross-domain',
    'lodo': 'LODO',
    'semantic_ood': 'Semantic OOD',
}
DOMAINS = ['Art', 'Clipart', 'Product', 'RealWorld']
MODES = ['linear_probe', 'partial_finetune', 'full_finetune']
MODE_LABEL = {'linear_probe': 'Linear', 'partial_finetune': 'Partial FT', 'full_finetune': 'Full FT'}
MODEL_ORDER = [
    'resnet50',
    'dinov3_convnext_tiny', 'dinov3_vit_s_plus',
    'dinov3_convnext_base', 'dinov3_vit_b',
    'dinov3_convnext_large', 'dinov3_vit_l',
]
MODEL_LABEL = {
    'resnet50': 'ResNet-50',
    'dinov3_convnext_tiny': 'ConvNeXt Tiny',
    'dinov3_vit_s_plus': 'ViT-S+',
    'dinov3_convnext_base': 'ConvNeXt Base',
    'dinov3_vit_b': 'ViT-B',
    'dinov3_convnext_large': 'ConvNeXt Large',
    'dinov3_vit_l': 'ViT-L',
}
MODEL_COLOR = {
    'resnet50': '#6B7280',
    'dinov3_convnext_tiny': '#2563EB',
    'dinov3_convnext_base': '#1D4ED8',
    'dinov3_convnext_large': '#1E40AF',
    'dinov3_vit_s_plus': '#DC2626',
    'dinov3_vit_b': '#B91C1C',
    'dinov3_vit_l': '#7F1D1D',
}
MODE_MARKER = {'linear_probe': 'o', 'partial_finetune': 's', 'full_finetune': '^'}
SELECTED_MODELS = ['resnet50', 'dinov3_convnext_base', 'dinov3_vit_b', 'dinov3_convnext_large', 'dinov3_vit_l']


def read_csv(path: Path) -> list[dict[str, str]]:
    with path.open('r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


def savefig(name: str) -> None:
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'saved {path}')


def mean(values):
    values = list(values)
    return float(np.mean(values)) if values else float('nan')


def metric_rows(metric: str, protocol: str | None = None) -> list[dict]:
    rows = [r for r in summary if r['metric'] == metric]
    if protocol:
        rows = [r for r in rows if r['protocol'] == protocol]
    return rows


def summary_value(protocol: str, model: str, mode: str, metric: str) -> float:
    vals = [float(r['mean']) for r in summary if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode and r['metric'] == metric]
    return mean(vals)

## Загрузка результатов

`acs_full.csv` содержит compact-view полного corruption score по всем 15 типам и 5 уровням severity. `summary.csv` используется для clean, calibration, OOD и compute-метрик.

In [ ]:
summary = read_csv(TABLES / 'summary.csv')
metrics_long = read_csv(TABLES / 'metrics_long.csv')
lodo_summary = read_csv(TABLES / 'lodo_summary.csv')
acs_runs = read_csv(TABLES / 'acs_full.csv')

for r in acs_runs:
    r['seed'] = int(r['seed'])
    r['clean_macro_f1'] = float(r['clean_macro_f1'])
    r['acs_macro_f1'] = float(r['acs_macro_f1'])
    r['relative_drop'] = float(r['relative_drop'])

for r in summary:
    r['mean'] = float(r['mean'])
    r['std'] = float(r['std'])
    r['n'] = int(r['n'])

print('summary rows:', len(summary))
print('metrics_long rows:', len(metrics_long))
print('acs rows:', len(acs_runs))
print('protocols:', {p: sum(1 for r in acs_runs if r['protocol'] == p) for p in PROTOCOLS})

## 1. Общая схема экспериментов

Сводка количества checkpoint по протоколам показывает, что анализ охватывает всю матрицу доменных и OOD-сценариев, а не только LODO.

In [ ]:
counts = defaultdict(int)
for r in acs_runs:
    counts[r['protocol']] += 1
fig, ax = plt.subplots(figsize=(7.5, 4.2))
labels = [PROTOCOL_LABEL[p] for p in PROTOCOLS]
values = [counts[p] for p in PROTOCOLS]
colors = ['#0F766E', '#9333EA', '#2563EB', '#DC2626']
ax.bar(labels, values, color=colors)
for i, v in enumerate(values):
    ax.text(i, v + 2, str(v), ha='center', va='bottom', fontweight='bold')
ax.set_title('Количество checkpoint по экспериментальным протоколам')
ax.set_ylabel('Checkpoint')
ax.set_ylim(0, max(values) * 1.18)
savefig('experiment_matrix_checkpoint_counts.png')

## 2. Clean Macro-F1 во всех протоколах

Heatmap показывает среднее clean-качество для каждой пары `модель × режим` внутри каждого протокола. Для `in_domain` и `lodo` значения усреднены по доменам.

In [ ]:
def clean_matrix(protocol: str) -> np.ndarray:
    out = np.full((len(MODEL_ORDER), len(MODES)), np.nan)
    for i, model in enumerate(MODEL_ORDER):
        for j, mode in enumerate(MODES):
            vals = [r['clean_macro_f1'] for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode]
            out[i, j] = mean(vals)
    return out

fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2), sharex=True, sharey=True)
for ax, protocol in zip(axes.flat, PROTOCOLS):
    data = clean_matrix(protocol)
    im = ax.imshow(data, cmap='RdYlGn', vmin=0.25, vmax=0.98, aspect='auto')
    ax.set_title(PROTOCOL_LABEL[protocol])
    ax.set_xticks(range(len(MODES)), [MODE_LABEL[m] for m in MODES], rotation=20, ha='right')
    ax.set_yticks(range(len(MODEL_ORDER)), [MODEL_LABEL[m] for m in MODEL_ORDER])
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if np.isfinite(data[i, j]):
                ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02, label='Macro-F1')
fig.suptitle('Clean Macro-F1 по протоколам, моделям и режимам', y=1.02, fontsize=14)
savefig('all_protocols_clean_macro_f1_heatmaps.png')

## 3. In-domain, Cross-domain и LODO: доменный сдвиг

График сопоставляет три доменных сценария. Он показывает цену перехода от совпадающего train/test домена к направленному переносу и LODO-обобщению.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14.2, 4.8), sharey=True)
for ax, protocol in zip(axes, ['in_domain', 'cross_domain', 'lodo']):
    rows = []
    for model in SELECTED_MODELS:
        for mode in MODES:
            vals = [r['clean_macro_f1'] for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode]
            rows.append((model, mode, mean(vals)))
    x = np.arange(len(SELECTED_MODELS))
    width = 0.24
    for k, mode in enumerate(MODES):
        vals = [v for m, mo, v in rows if mo == mode]
        ax.bar(x + (k - 1) * width, vals, width=width, label=MODE_LABEL[mode])
    ax.set_title(PROTOCOL_LABEL[protocol])
    ax.set_xticks(x, [MODEL_LABEL[m] for m in SELECTED_MODELS], rotation=25, ha='right')
    ax.set_ylim(0.2, 1.0)
    ax.set_ylabel('Macro-F1')
axes[0].legend()
fig.suptitle('Clean Macro-F1: in-domain, cross-domain и LODO', y=1.04, fontsize=14)
savefig('domain_protocols_clean_macro_f1.png')

## 4. Подробно по доменам: In-domain и LODO

Две тепловые карты показывают, какие домены системно проще/сложнее. Для компактности используется лучший режим каждой модели по среднему значению внутри соответствующего протокола.

In [ ]:
def best_mode_for(protocol: str, model: str) -> str:
    scores = []
    for mode in MODES:
        vals = [r['clean_macro_f1'] for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode]
        scores.append((mean(vals), mode))
    return max(scores)[1]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2), sharey=True)
for ax, protocol in zip(axes, ['in_domain', 'lodo']):
    data = []
    mode_labels = []
    for model in MODEL_ORDER:
        mode = best_mode_for(protocol, model)
        mode_labels.append(MODE_LABEL[mode])
        row = []
        for domain in DOMAINS:
            vals = [r['clean_macro_f1'] for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode and r['domain'] == domain]
            row.append(mean(vals))
        data.append(row)
    data = np.array(data)
    im = ax.imshow(data, cmap='RdYlGn', vmin=0.25, vmax=0.98, aspect='auto')
    ax.set_title(f'{PROTOCOL_LABEL[protocol]}: best mode per model')
    ax.set_xticks(range(len(DOMAINS)), DOMAINS)
    ax.set_yticks(range(len(MODEL_ORDER)), [f'{MODEL_LABEL[m]} ({mode_labels[i]})' for i, m in enumerate(MODEL_ORDER)])
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02, label='Macro-F1')
fig.suptitle('Доменная структура качества: in-domain и held-out LODO', y=1.03, fontsize=14)
savefig('domain_level_clean_heatmaps.png')

## 5. Масштабирование CNN и ViT

График показывает разницу ViT-CNN внутри уровней Small/Base/Large. Положительное значение означает преимущество ViT.

In [ ]:
pairs = [
    ('Small', 'dinov3_convnext_tiny', 'dinov3_vit_s_plus'),
    ('Base', 'dinov3_convnext_base', 'dinov3_vit_b'),
    ('Large', 'dinov3_convnext_large', 'dinov3_vit_l'),
]
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8), sharey=True)
for ax, metric, title in zip(axes, ['clean_macro_f1', 'acs_macro_f1'], ['Clean Macro-F1', 'Full ACS Macro-F1']):
    labels, vals, colors = [], [], []
    for protocol in PROTOCOLS:
        for level, cnn, vit in pairs:
            c = mean(r[metric] for r in acs_runs if r['protocol'] == protocol and r['model'] == cnn)
            v = mean(r[metric] for r in acs_runs if r['protocol'] == protocol and r['model'] == vit)
            labels.append(f'{PROTOCOL_LABEL[protocol]}\n{level}')
            vals.append(v - c)
            colors.append('#B91C1C' if v >= c else '#1D4ED8')
    x = np.arange(len(vals))
    ax.axhline(0, color='#111827', linewidth=0.9)
    ax.bar(x, vals, color=colors)
    ax.set_title(title)
    ax.set_xticks(x, labels, rotation=55, ha='right')
    ax.set_ylabel('ViT - CNN')
fig.suptitle('Архитектурная разница внутри fair-уровней Small/Base/Large', y=1.03, fontsize=14)
savefig('scaling_vit_minus_cnn_by_protocol.png')

## 6. Clean качество vs robustness к corruptions

Каждая точка — среднее по доменам/запускам для пары `модель × режим` внутри протокола. Правый верхний угол соответствует одновременно высокому clean-качеству и высокой устойчивости.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.8), sharex=True, sharey=True)
for ax, protocol in zip(axes.flat, PROTOCOLS):
    for model in MODEL_ORDER:
        for mode in MODES:
            rows = [r for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode]
            if not rows:
                continue
            ax.scatter(
                mean(r['clean_macro_f1'] for r in rows),
                mean(r['acs_macro_f1'] for r in rows),
                s=75,
                marker=MODE_MARKER[mode],
                color=MODEL_COLOR[model],
                edgecolor='white', linewidth=0.8, alpha=0.9,
            )
    ax.plot([0.2, 1.0], [0.2, 1.0], color='#9CA3AF', linestyle='--', linewidth=1)
    ax.set_title(PROTOCOL_LABEL[protocol])
    ax.set_xlabel('Clean Macro-F1')
    ax.set_ylabel('ACS Macro-F1')
    ax.set_xlim(0.25, 0.98)
    ax.set_ylim(0.15, 0.95)
model_handles = [Line2D([0], [0], marker='o', color='w', label=MODEL_LABEL[m], markerfacecolor=MODEL_COLOR[m], markersize=8) for m in MODEL_ORDER]
mode_handles = [Line2D([0], [0], marker=MODE_MARKER[m], color='#111827', linestyle='None', label=MODE_LABEL[m], markersize=8) for m in MODES]
axes[1, 1].legend(handles=model_handles, loc='lower right', fontsize=7, title='Model')
axes[0, 0].legend(handles=mode_handles, loc='upper left', fontsize=8, title='Mode')
fig.suptitle('Clean Macro-F1 vs полный ACS во всех протоколах', y=1.02, fontsize=14)
savefig('all_protocols_clean_vs_acs.png')

## 7. Relative Corruption Drop по протоколам

Чем меньше падение, тем лучше модель сохраняет качество под искажениями.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8.5), sharex=True)
for ax, protocol in zip(axes.flat, PROTOCOLS):
    rows = []
    for model in SELECTED_MODELS:
        for mode in MODES:
            vals = [r['relative_drop'] for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode]
            rows.append((model, mode, 100 * mean(vals)))
    rows = sorted(rows, key=lambda x: x[2], reverse=True)
    y = np.arange(len(rows))
    ax.barh(y, [v for _, _, v in rows], color=[MODEL_COLOR[m] for m, _, _ in rows])
    ax.set_yticks(y, [f'{MODEL_LABEL[m]} / {MODE_LABEL[mo]}' for m, mo, _ in rows])
    ax.invert_yaxis()
    ax.set_title(PROTOCOL_LABEL[protocol])
    ax.set_xlabel('Relative drop, %')
fig.suptitle('Падение Macro-F1 на полном наборе corruptions', y=1.02, fontsize=14)
savefig('all_protocols_relative_corruption_drop.png')

## 8. Деградация по severity

Кривые показывают форму падения качества от слабых к сильным искажениям. Используется linear probing как наиболее чистая оценка качества признаков.

In [ ]:
def load_severity_rows() -> list[dict]:
    rows = []
    model_order = MODEL_ORDER
    for run_dir in sorted(p for p in METRICS.iterdir() if p.is_dir()):
        run = run_dir.name
        model = next((m for m in model_order if f'_{m}_' in run), None)
        mode = next((m for m in MODES if f'_{m}_' in run), None)
        if not model or not mode:
            continue
        if '_lodo_' in run:
            protocol = 'lodo'
        elif '_cross_domain_' in run:
            protocol = 'cross_domain'
        elif '_in_domain_' in run:
            protocol = 'in_domain'
        elif '_semantic_ood_' in run:
            protocol = 'semantic_ood'
        else:
            continue
        for name in ['corruptions_s1_s2_s4.csv', 'corruptions_s3_s5.csv']:
            path = run_dir / name
            if not path.exists():
                continue
            for r in read_csv(path):
                if r['severity'] == '0':
                    continue
                rows.append({
                    'protocol': protocol,
                    'model': model,
                    'mode': mode,
                    'severity': int(r['severity']),
                    'corruption': r['corruption'],
                    'macro_f1': float(r['macro_f1']),
                })
    return rows

severity_rows = load_severity_rows()
fig, axes = plt.subplots(2, 2, figsize=(11.5, 8.2), sharex=True, sharey=True)
for ax, protocol in zip(axes.flat, PROTOCOLS):
    for model in SELECTED_MODELS:
        vals = []
        for severity in [1, 2, 3, 4, 5]:
            subset = [r['macro_f1'] for r in severity_rows if r['protocol'] == protocol and r['mode'] == 'linear_probe' and r['model'] == model and r['severity'] == severity]
            vals.append(mean(subset))
        ax.plot([1, 2, 3, 4, 5], vals, marker='o', linewidth=2, color=MODEL_COLOR[model], label=MODEL_LABEL[model])
    ax.set_title(PROTOCOL_LABEL[protocol])
    ax.set_xlabel('Severity')
    ax.set_ylabel('Macro-F1')
    ax.set_xticks([1, 2, 3, 4, 5])
    ax.set_ylim(0.15, 0.95)
axes[0, 0].legend(fontsize=8)
fig.suptitle('Linear Probe: деградация качества по severity', y=1.02, fontsize=14)
savefig('all_protocols_severity_curves_linear.png')

## 9. Corruption types: какие искажения наиболее сложные

Heatmap усредняет full ACS по моделям и режимам внутри каждого протокола для каждого типа искажения.

In [ ]:
corruptions = [
    'gaussian_noise', 'shot_noise', 'impulse_noise',
    'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur',
    'snow', 'frost', 'fog', 'brightness',
    'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression',
]
data = []
for protocol in PROTOCOLS:
    row = []
    for corruption in corruptions:
        vals = [r['macro_f1'] for r in severity_rows if r['protocol'] == protocol and r['corruption'] == corruption]
        row.append(mean(vals))
    data.append(row)
data = np.array(data)
fig, ax = plt.subplots(figsize=(12.8, 3.8))
im = ax.imshow(data, cmap='RdYlGn', vmin=0.25, vmax=0.92, aspect='auto')
ax.set_title('Средний Macro-F1 по типам corruptions и протоколам')
ax.set_xticks(range(len(corruptions)), [c.replace('_', '\n') for c in corruptions])
ax.set_yticks(range(len(PROTOCOLS)), [PROTOCOL_LABEL[p] for p in PROTOCOLS])
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        ax.text(j, i, f'{data[i, j]:.2f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01, label='Macro-F1')
savefig('corruption_type_by_protocol_heatmap.png')

## 10. Calibration: ECE по протоколам

ECE отражает качество confidence-оценок. Для `in_domain` и `semantic_ood` используется `id_test_ece`, для `cross_domain` и `lodo` — `target_test_ece`.

In [ ]:
def ece_metric(protocol: str) -> str:
    return 'target_test_ece' if protocol in {'cross_domain', 'lodo'} else 'id_test_ece'

fig, axes = plt.subplots(2, 2, figsize=(12, 8.0), sharey=True)
for ax, protocol in zip(axes.flat, PROTOCOLS):
    metric = ece_metric(protocol)
    rows = []
    for model in SELECTED_MODELS:
        for mode in MODES:
            vals = [r['mean'] for r in summary if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode and r['metric'] == metric]
            rows.append((model, mode, mean(vals)))
    rows = sorted(rows, key=lambda x: x[2])
    y = np.arange(len(rows))
    ax.barh(y, [v for _, _, v in rows], color=[MODEL_COLOR[m] for m, _, _ in rows])
    ax.set_yticks(y, [f'{MODEL_LABEL[m]} / {MODE_LABEL[mo]}' for m, mo, _ in rows])
    ax.invert_yaxis()
    ax.set_title(PROTOCOL_LABEL[protocol])
    ax.set_xlabel('ECE')
fig.suptitle('Калибровка вероятностей: Expected Calibration Error', y=1.02, fontsize=14)
savefig('calibration_ece_by_protocol.png')

## 11. Semantic OOD: MSP и Energy

AUROC выше — лучше. FPR@95TPR ниже — лучше. Эти графики показывают поведение моделей на unknown-классах.

In [ ]:
def ood_matrix(metric: str) -> np.ndarray:
    data = np.full((len(MODEL_ORDER), len(MODES)), np.nan)
    for i, model in enumerate(MODEL_ORDER):
        for j, mode in enumerate(MODES):
            vals = [r['mean'] for r in summary if r['protocol'] == 'semantic_ood' and r['model'] == model and r['mode'] == mode and r['metric'] == metric]
            data[i, j] = mean(vals)
    return data

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), sharey=True)
for ax, metric, title, vmin, vmax in [
    (axes[0], 'ood/msp_auroc', 'MSP AUROC ↑', 0.5, 1.0),
    (axes[1], 'ood/energy_auroc', 'Energy AUROC ↑', 0.5, 1.0),
]:
    data = ood_matrix(metric)
    im = ax.imshow(data, cmap='RdYlGn', vmin=vmin, vmax=vmax, aspect='auto')
    ax.set_title(title)
    ax.set_xticks(range(len(MODES)), [MODE_LABEL[m] for m in MODES], rotation=20, ha='right')
    ax.set_yticks(range(len(MODEL_ORDER)), [MODEL_LABEL[m] for m in MODEL_ORDER])
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if np.isfinite(data[i, j]):
                ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
fig.suptitle('Semantic OOD: качество отделения unknown-классов', y=1.03, fontsize=14)
savefig('semantic_ood_auroc_heatmaps.png')

fig, axes = plt.subplots(1, 2, figsize=(11.5, 5.2), sharey=True)
for ax, metric, title in [
    (axes[0], 'ood/msp_fpr95tpr', 'MSP FPR@95TPR ↓'),
    (axes[1], 'ood/energy_fpr95tpr', 'Energy FPR@95TPR ↓'),
]:
    data = ood_matrix(metric)
    im = ax.imshow(data, cmap='RdYlGn_r', vmin=0.0, vmax=1.0, aspect='auto')
    ax.set_title(title)
    ax.set_xticks(range(len(MODES)), [MODE_LABEL[m] for m in MODES], rotation=20, ha='right')
    ax.set_yticks(range(len(MODEL_ORDER)), [MODEL_LABEL[m] for m in MODEL_ORDER])
    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            if np.isfinite(data[i, j]):
                ax.text(j, i, f'{data[i, j]:.3f}', ha='center', va='center', fontsize=7)
fig.colorbar(im, ax=axes.ravel().tolist(), fraction=0.025, pad=0.02)
fig.suptitle('Semantic OOD: ложные срабатывания при 95% TPR', y=1.03, fontsize=14)
savefig('semantic_ood_fpr95_heatmaps.png')

## 12. Вычислительная эффективность

Scatter показывает trade-off между качеством и количеством параметров. Размер точки пропорционален throughput.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0), sharey=True)
for ax, protocol in zip(axes, ['in_domain', 'lodo']):
    for model in MODEL_ORDER:
        for mode in MODES:
            clean_vals = [r['clean_macro_f1'] for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode]
            if not clean_vals:
                continue
            params = summary_value(protocol, model, mode, 'compute_params') / 1e6
            throughput = summary_value(protocol, model, mode, 'compute_throughput_img_sec')
            size = 25 + min(max(throughput, 1), 900) / 900 * 120
            ax.scatter(params, mean(clean_vals), s=size, marker=MODE_MARKER[mode], color=MODEL_COLOR[model], edgecolor='white', linewidth=0.8, alpha=0.85)
    ax.set_xscale('log')
    ax.set_title(PROTOCOL_LABEL[protocol])
    ax.set_xlabel('Parameters, million, log scale')
    ax.set_ylabel('Clean Macro-F1')
    ax.set_ylim(0.2, 1.0)
axes[0].legend(handles=[Line2D([0],[0],marker=MODE_MARKER[m],color='#111827',linestyle='None',label=MODE_LABEL[m],markersize=8) for m in MODES], loc='lower right')
fig.suptitle('Качество и вычислительная стоимость', y=1.03, fontsize=14)
savefig('quality_vs_params_tradeoff.png')

## 13. Сводное ранжирование

Печатаются лучшие конфигурации по clean Macro-F1 и ACS в каждом протоколе.

In [ ]:
for protocol in PROTOCOLS:
    print('\n' + PROTOCOL_LABEL[protocol])
    grouped = []
    for model in MODEL_ORDER:
        for mode in MODES:
            rows = [r for r in acs_runs if r['protocol'] == protocol and r['model'] == model and r['mode'] == mode]
            if not rows:
                continue
            grouped.append({
                'model': MODEL_LABEL[model],
                'mode': MODE_LABEL[mode],
                'clean': mean(r['clean_macro_f1'] for r in rows),
                'acs': mean(r['acs_macro_f1'] for r in rows),
                'drop': 100 * mean(r['relative_drop'] for r in rows),
            })
    for r in sorted(grouped, key=lambda x: x['clean'], reverse=True)[:5]:
        print(f"clean: {r['model']:16s} {r['mode']:10s} F1={r['clean']:.3f} ACS={r['acs']:.3f} drop={r['drop']:.1f}%")
    for r in sorted(grouped, key=lambda x: x['acs'], reverse=True)[:3]:
        print(f"  ACS: {r['model']:16s} {r['mode']:10s} F1={r['clean']:.3f} ACS={r['acs']:.3f} drop={r['drop']:.1f}%")